# Correlations & Diversification

Two portfolios with the same volatility can carry wildly different *risk* depending on how
their holdings move together. This is the "am I actually diversified, or do I own ten things
that are really one bet?" notebook.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} across {positions['account_name'].nunique()} accounts "
      f"· {len(history)} trading days {history.index.min():%Y-%m-%d}→{history.index.max():%Y-%m-%d}")

In [ ]:
returns = A.holding_returns(positions, history)
weights = A.portfolio_weights(positions)
port_ret = A.portfolio_return_series(positions, history)
print(f"{returns.shape[1]} priced holdings · {returns.shape[0]} daily returns")

## Correlation heatmap

Daily-return correlation between holdings, reordered so similar names cluster together.
Deep red blocks are groups that rise and fall as one — diversification you *think* you have
but don't.

In [ ]:
co = A.correlation_ordered(returns)
fig = px.imshow(co, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                aspect="auto", title="Holding return correlation (clustered)")
fig.update_layout(height=620, coloraxis_colorbar_title="ρ")
fig.show()

## Diversification scorecard

**Diversification ratio** = weighted-average holding volatility ÷ portfolio volatility. At
1.0 your holdings move in lockstep (no benefit); higher means correlations are cancelling
risk for you. Shown next to the average pairwise correlation.

In [ ]:
dr = A.diversification_ratio(weights, returns)
avg_corr = co.where(~np.eye(len(co), dtype=bool)).stack().mean()
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "indicator"}, {"type": "indicator"}]])
fig.add_trace(go.Indicator(mode="gauge+number", value=dr, title={"text": "Diversification ratio"},
    number={"valueformat": ".2f"}, gauge={"axis": {"range": [1, 2.5]},
    "bar": {"color": PALETTE[0]}}), row=1, col=1)
fig.add_trace(go.Indicator(mode="gauge+number", value=avg_corr, title={"text": "Avg pairwise ρ"},
    number={"valueformat": ".2f"}, gauge={"axis": {"range": [-0.2, 1]},
    "bar": {"color": PALETTE[1]}}), row=1, col=2)
fig.update_layout(height=300)
fig.show()

## Where does the risk actually come from?

Each holding's **share of total portfolio variance** next to its **share of capital**. When
the risk bar towers over the weight bar, that name is punching above its size — a small,
volatile, highly-correlated position can dominate your risk while looking minor on an
allocation chart.

In [ ]:
rc = A.risk_contributions(weights, returns)
fig = go.Figure()
fig.add_bar(x=rc.index, y=rc["weight"], name="capital weight", marker_color=PALETTE[2])
fig.add_bar(x=rc.index, y=rc["risk_contribution"], name="risk contribution", marker_color=PALETTE[3])
fig.update_layout(barmode="group", height=440, yaxis_tickformat=".0%",
                  title="Risk contribution vs capital weight", xaxis_title="",
                  hovermode="x unified")
fig.show()
display(rc.style.format({"weight": "{:.1%}", "vol": "{:.1%}", "risk_contribution": "{:.1%}"}))

## Rolling correlation to the market

90-day correlation of the portfolio's daily returns to the S&P 500. Spikes toward 1.0 are
exactly when diversification fails you — in sell-offs, things you thought were independent
start moving together.

In [ ]:
spy_ret = history["SPY"].pct_change()
rc90 = port_ret.rolling(90).corr(spy_ret).dropna()
fig = go.Figure(go.Scatter(x=rc90.index, y=rc90, fill="tozeroy", line=dict(color=PALETTE[0])))
fig.add_hline(y=rc90.mean(), line_dash="dot", line_color="grey",
              annotation_text=f"avg {rc90.mean():.2f}")
fig.update_layout(height=380, yaxis_range=[0, 1], title="Rolling 90-day correlation to S&P 500",
                  hovermode="x")
fig.show()

## Which holdings actually diversify you?

Each holding's correlation to the *rest of the portfolio* against its weight. Names low on
the y-axis are your genuine diversifiers; high-weight, high-correlation names (top-right)
are where concentration risk really lives.

In [ ]:
corr_to_port = returns.apply(lambda s: s.corr(port_ret.reindex(s.index)))
div = pd.DataFrame({"corr_to_portfolio": corr_to_port, "weight": weights}).dropna()
div["asset_class"] = positions.groupby("symbol")["asset_class"].first()
fig = px.scatter(div.rename_axis("symbol").reset_index(), x="weight", y="corr_to_portfolio", size="weight",
                 color="asset_class", hover_name="symbol", size_max=50,
                 color_discrete_sequence=PALETTE, title="Diversifiers vs concentration risk")
fig.add_hline(y=div["corr_to_portfolio"].mean(), line_dash="dot", line_color="grey")
fig.update_layout(height=480, xaxis_tickformat=".0%", xaxis_title="portfolio weight",
                  yaxis_title="correlation to rest of portfolio")
fig.show()

---
*Correlations use the priced/​proxied sleeve and the full window; they shift over time
(see the rolling chart) and rise in stress. A low pairwise number across a few names is
weaker evidence of diversification than it looks.*